# Giving an LLM tools: a buy/hold/sell recommendation

A language model on its own can only tell you what it absorbed during training. Ask it
about a company and it will answer confidently, in the voice of an analyst, using
numbers it half-remembers from a world that has since moved on.

An **agent** is the fix, and it is a small idea: let the model call functions. We give it
a loop. Each time round, the model either answers or asks to run one of our functions.
We run it, hand back the result, and ask again.

Today we build that up in four stages, and at each stage we look at what the answer is
still missing:

| Stage | What the model can reach | What is still wrong |
|---|---|---|
| 1 | nothing | invented figures |
| 2 | a calculator | precise arithmetic on invented figures |
| 3 | web search | current context, but as prose |
| 4 | financial data | figures you can check |

Run the cells in order. At the end: a few bonus experiments, and one job of your own, which is to add a tool.

## Section 0 · Setup

Run the next three cells once: install the libraries, read your credentials, and create
the client. If a cell stops with a message about the API key, the message says how to fix it.

In [ ]:
# Colab starts empty every session, so install what we need.
# The versions are pinned deliberately: these are the ones this notebook was
# built and tested against, and yfinance in particular changes often enough
# that an unpinned install could behave differently from one week to the next.
!pip install -q openai==3.42.0 yfinance==1.7.0 ddgs==9.16.1

In [14]:
import getpass
import importlib
import os
from pathlib import Path

DEFAULT_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL = "gpt-4o-mini"


class CredentialError(Exception):
    """Raised when credentials are present but the provider rejects them."""

    @classmethod
    def unauthorized(cls) -> "CredentialError":
        return cls(
            "The API key was rejected. Check that you pasted the whole key, then "
            "re-enter it: open the key icon in the left sidebar to fix the Colab "
            "secret, or use Runtime -> Restart session and run the cells again."
        )

    @classmethod
    def unknown_model(cls, model: str | None = None) -> "CredentialError":
        return cls(
            f"The provider does not recognise the model {model!r}. Model names get "
            "retired: set a MODEL secret (key icon in the left sidebar) to one your "
            "account has, then use Runtime -> Restart session."
        )

    @classmethod
    def unreachable(cls, base_url: str | None = None) -> "CredentialError":
        return cls(
            f"Could not reach the model provider at {base_url!r}. Check the BASE_URL "
            "secret. Note that an address on a private network -- anything like "
            "192.168.x.x, 10.x.x.x or localhost -- is not reachable from Colab, "
            "because this notebook runs on a Google server rather than on your machine."
        )

    @classmethod
    def rate_limited(cls) -> "CredentialError":
        return cls(
            "The provider is rate-limiting this key -- usually because everyone in "
            "the room is calling it at once. Wait a few seconds and run the cell again."
        )


def _from_colab(name: str) -> str | None:
    """Read a Colab secret, or None when not on Colab / not set / not granted."""
    try:
        userdata = importlib.import_module("google.colab.userdata")
        value = userdata.get(name)
    except Exception:
        return None
    return value or None


def _load_dotenv(path: str = ".env") -> None:
    """Copy `KEY=value` lines from a .env file into the environment, if one exists.

    Colab has no .env file, so this does nothing there -- it is what makes the same
    notebook run locally, where a .env is the usual place to keep these. Values
    already in the environment win, so an explicit export still overrides the file.
    """
    try:
        text = Path(path).read_text(encoding="utf-8")
    except OSError:
        return
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


def _lookup(name: str) -> str | None:
    """Colab secret first, then the environment (which includes any .env file)."""
    return _from_colab(name) or os.getenv(name) or None


def get_credentials() -> tuple[str, str, str]:
    """Return (base_url, api_key, model), asking for the key only if nothing has it.

    Each value is looked for in the same order: a Colab secret, then the
    environment (including a .env file if one is present), then the default
    above. Only the API key has no default, so only the API key can prompt.
    """
    _load_dotenv()
    base_url = _lookup("BASE_URL") or DEFAULT_BASE_URL
    model = _lookup("MODEL") or DEFAULT_MODEL
    api_key = _lookup("OPENAI_API_KEY")
    if not api_key:
        api_key = getpass.getpass("Paste the API key for this session: ").strip()
    return base_url, api_key, model


BASE_URL, API_KEY, MODEL = get_credentials()
print(f"Using model {MODEL} at {BASE_URL}")

Using model openai/gpt-4o at https://openrouter.ai/api/v1


In [15]:
from openai import OpenAI

# One client, pointed at whichever provider BASE_URL names.
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


# The slides show the naive version: catch everything, print a message. It hides the
# real error from the loop. The version below translates the common failures instead.
# def _create(client, **kwargs):
#     try:
#         return client.chat.completions.create(**kwargs)
#     except Exception as exc:
#         return f"unexpected error: {exc}"


def _base_url_of(client) -> str | None:
    """Best-effort read of the client's configured base URL, for error messages."""
    return str(getattr(client, "base_url", "") or "") or None


def _create(client, **kwargs):
    """Call the model, turning a rejected key into something a student can act on.

    The provider raises a 401 as a long traceback naming headers and request ids.
    A finance student reading that has no idea the fix is "re-paste the key", so
    we translate it once, here, where every model call passes through.
    """
    try:
        return client.chat.completions.create(**kwargs)
    except Exception as exc:
        status = getattr(exc, "status_code", None)
        if status == 401 or "authentication" in type(exc).__name__.lower():
            raise CredentialError.unauthorized() from exc
        if status == 404:
            raise CredentialError.unknown_model(kwargs.get("model")) from exc
        if status == 429:
            raise CredentialError.rate_limited() from exc
        name = type(exc).__name__.lower()
        if "connection" in name or "timeout" in name:
            raise CredentialError.unreachable(kwargs.get("base_url") or _base_url_of(client)) from exc
        raise

In [16]:
COMPANY = "Roche Holding AG (ROP.SW)"
TICKER = "ROP.SW"

SYSTEM_PROMPT = "You are a stock analyst."
NAIVE_SYSTEM_PROMPT = SYSTEM_PROMPT  # to switch back after the bonus cells


def _user_message(company: str) -> str:
    """What we actually ask for. Kept apart from SYSTEM_PROMPT, which is fixed."""
    return f"Give a buy/hold/sell recommendation for {company} with a short justification."


BANK_SYSTEM_PROMPT = """ROLE
You are a research assistant at a private bank. You prepare
stock briefings for client advisors, not for clients.

GOAL
Answer the advisor's question in a one-page briefing.

TOOLS
Use web_search for news, get_stock_data for prices and
dividends, calculator for every number you compute.

RULES
- Never calculate in your head. Always use the calculator.
- Cite the source and date of every figure.
- If data is missing or older than 7 days, say so.
- Do not give a buy/sell recommendation.

OUTPUT
Summary (3 sentences) · Key figures (table) · Risks · Sources"""

## Section 1 · The loop and how to read its trace

Every stage below prints the same three things after the answer:

- **Tool calls, in order**: which function the model asked for, with which arguments.
  `get_stock_data(ticker='ROP.SW')` is a different request from `get_stock_data(ticker='Roche')`.
- **Model calls**: how many times the loop went round. Each one is a paid API call.
- **Tokens**: the size of everything sent and received, added up over those calls.

Judge the *trace*, not only the answer: a confident recommendation with no tool calls
behind it is a recollection.

In [17]:
import json
from dataclasses import dataclass, field


@dataclass
class AgentResult:
    answer: str
    model_calls: int
    tool_calls: list[str]
    stopped_early: bool
    # One entry per tool call: {"round", "tool", "arguments", "result"}.
    trace: list[dict] = field(default_factory=list)
    total_tokens: int = 0


def _arguments_of(call):
    """The arguments the model sent, as a dict -- or the raw text if it is not JSON."""
    raw = call.function.arguments or "{}"
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return raw


def _invoke(call, registry: dict) -> str:
    """Run one tool call and return its result as text. Never raises.

    A tool that raised would end the conversation instead of informing it, so every
    failure here becomes a message the model can read and react to -- which is what
    lets a student's half-finished tool fail without killing the cell.
    """
    name = call.function.name
    function = registry.get(name)
    if function is None:
        return f"Error: unknown tool {name}"
    try:
        arguments = json.loads(call.function.arguments or "{}")
    except json.JSONDecodeError as exc:
        return f"Error: could not read the arguments for {name}: {exc}"
    try:
        return str(function(**arguments))
    except Exception as exc:
        return f"Error: {exc}"


def run_agent(
    client,
    model: str,
    company: str,
    schemas: list[dict],
    registry: dict,
    max_rounds: int = 8,
    question: str | None = None,
) -> AgentResult:
    """Stages 2-4: call the model, run any tool it asks for, repeat until it answers."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question or _user_message(company)},
    ]
    model_calls = 0
    total_tokens = 0
    called: list[str] = []
    trace: list[dict] = []

    for _ in range(max_rounds):
        # Leave the key out entirely when there are no tools: some providers reject
        # "tools": null.
        extra = {"tools": schemas} if schemas else {}
        response = _create(client, model=model, messages=messages, **extra)
        model_calls += 1
        total_tokens += getattr(getattr(response, "usage", None), "total_tokens", 0) or 0
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return AgentResult(
                message.content, model_calls, called, False, trace, total_tokens
            )

        for call in message.tool_calls:
            called.append(call.function.name)
            result = _invoke(call, registry)
            trace.append(
                {
                    "round": model_calls,
                    "tool": call.function.name,
                    "arguments": _arguments_of(call),
                    "result": result,
                }
            )
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    # The model kept asking for tools and never settled. Stop rather than spend the
    # shared key on a loop that is not converging. The default of 8 (slide 37) is a
    # deliberate cap; in one live run a search-happy model used 6 rounds on stage 3 alone and was
    # cut off mid-answer.
    last = next(
        (m.content for m in reversed(messages) if getattr(m, "content", None)),
        None,
    )
    return AgentResult(
        last or f"Stopped after {max_rounds} rounds without a final answer.",
        model_calls,
        called,
        True,
        trace,
        total_tokens,
    )


def format_trace(result: AgentResult) -> str:
    """One line per tool call, in order: `1. get_stock_data(ticker='ROP.SW')`."""
    lines = []
    for number, step in enumerate(result.trace, start=1):
        args = step["arguments"]
        shown = ", ".join(f"{k}={v!r}" for k, v in args.items()) if isinstance(args, dict) else args
        lines.append(f"{number}. {step['tool']}({shown})")
    return "\n".join(lines)


def run_stage(
    client,
    model: str,
    company: str,
    schemas: list[dict],
    registry: dict,
    runs: int = 1,
    question: str | None = None,
):
    """Run one stage `runs` times, each from a clean start, and print answer and trace."""
    results = []
    for number in range(1, runs + 1):
        if runs > 1:
            print(f"--- Run {number} of {runs} ---")
        result = run_agent(client, model, company, schemas, registry, question=question)
        print(result.answer)
        print("\nTool calls, in order:")
        print(format_trace(result) or "(none)")
        print(f"\n--- {result.model_calls} model calls, {result.total_tokens} tokens")
        if result.stopped_early:
            print("--- NOTE: hit the round limit, so the answer above is incomplete.")
        results.append(result)
    return results

### What to write down

For each stage, one row. In stage 1 run the cell three times and note all three recommendations.

| Stage | Quoted price (as of) | Actual price | Figures with a source | Recommendation, run 1 / 2 / 3 | Consistent? | Tool calls | Tokens |
|---|---|---|---|---|---|---|---|
| 1 · only LLM | | | | | | | |
| 2 · + calculator | | | | | | | |
| 3 · + web search | | | | | | | |
| 4 · + yfinance | | | | | | | |

Look at the trace, not only the answer: which tools, with which arguments, in which order?

## Stage 1 · Only the LLM

No tools. What can go wrong?

In [18]:
# Stage 1: the model alone. Same loop, empty tool list. Run three times: the answers differ.
run_stage(client, MODEL, COMPANY, [], {}, runs=3)

--- Run 1 of 3 ---
As of the latest update, I do not have real-time data, so I cannot provide current stock recommendations or the latest market conditions for Roche Holding AG (ROP.SW). However, I can guide you on how to approach this:

1. **Financial Performance**: Look for Roche's recent financial statements, including revenue growth, profit margins, and cash flow. Consistent growth and strong profitability could indicate a positive outlook.

2. **Pipeline and Innovation**: Roche, being a pharmaceutical company, heavily relies on its drug pipeline. Assess the status of their pipeline products, recent drug approvals, and R&D developments.

3. **Market Conditions**: Analyze the overall pharmaceutical and biotechnology market conditions, including regulatory news and patent expirations which may affect Roche's performance.

4. **Competitor Analysis**: Compare Roche's position with its peers in the industry. This includes both market share and innovation leadership.

5. **Macroeconomic 

[AgentResult(answer="As of the latest update, I do not have real-time data, so I cannot provide current stock recommendations or the latest market conditions for Roche Holding AG (ROP.SW). However, I can guide you on how to approach this:\n\n1. **Financial Performance**: Look for Roche's recent financial statements, including revenue growth, profit margins, and cash flow. Consistent growth and strong profitability could indicate a positive outlook.\n\n2. **Pipeline and Innovation**: Roche, being a pharmaceutical company, heavily relies on its drug pipeline. Assess the status of their pipeline products, recent drug approvals, and R&D developments.\n\n3. **Market Conditions**: Analyze the overall pharmaceutical and biotechnology market conditions, including regulatory news and patent expirations which may affect Roche's performance.\n\n4. **Competitor Analysis**: Compare Roche's position with its peers in the industry. This includes both market share and innovation leadership.\n\n5. **Ma

## What just went wrong

Read the answer again, and this time check it.

**Find every number it gave you** -- the price, the P/E, the market cap -- and look them
up. Compare notes with the person next to you: you are probably holding different
numbers, and none of them are today's.

The model is not lying. It is doing the only thing it can: recalling. Its training data
stopped at some point in the past, and it has no way to know that, so it reports old
figures in the present tense and sounds exactly as certain either way.

Two things make this sharper for Roche specifically:

- **Currency and share type.** Roche trades in Swiss francs (CHF), not dollars, and there
  are two listed securities: the Genussschein (`ROG`) and the Inhaberaktie (`RO`), at
  different prices. A model that quotes a price without saying which one, or in which
  currency, has told you nothing you can check.
- **It reads well.** Roche is one of the most written-about companies in Europe, so the
  *narrative* the model produces is genuinely competent. Only the figures are wrong. That
  combination -- sound reasoning over broken inputs -- is the dangerous one, and it is
  much harder to spot than obvious nonsense.

No amount of rewording the prompt fixes this. You cannot ask a model to know something it
was never told. It needs a way to go and look.

## Stage 2 · + calculator

Same prompt, same model, same loop. Only the tool list changes. Does the answer get better?

In [19]:
import ast

_ALLOWED_NODES = (
    ast.Expression,
    ast.Constant,
    ast.UnaryOp,
    ast.BinOp,
    ast.USub,
    ast.UAdd,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.Mod,
    ast.Pow,
)

_MAX_EXPONENT = 1000

_BINARY_OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
    ast.FloorDiv: lambda a, b: a // b,
    ast.Mod: lambda a, b: a % b,
    ast.Pow: lambda a, b: a**b,
}


def _evaluate(node):
    """Walk the validated tree and compute its value."""
    if isinstance(node, ast.Expression):
        return _evaluate(node.body)
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.UnaryOp):
        value = _evaluate(node.operand)
        return -value if isinstance(node.op, ast.USub) else +value
    if isinstance(node, ast.BinOp):
        left, right = _evaluate(node.left), _evaluate(node.right)
        if isinstance(node.op, ast.Pow) and right > _MAX_EXPONENT:
            raise ValueError(f"exponent above {_MAX_EXPONENT} is too large to compute")
        return _BINARY_OPS[type(node.op)](left, right)
    raise ValueError("unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. "1234.5 * 0.21"."""
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        return f"Error: {expression!r} is not a valid arithmetic expression."

    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            return (
                f"Error: {expression!r} contains something that is not arithmetic. "
                "Pass numbers and the operators + - * / // % ** only."
            )
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
            return f"Error: {node.value!r} is not a number."
        if isinstance(node, ast.Constant) and isinstance(node.value, bool):
            return f"Error: {node.value!r} is not a number."

    try:
        return str(_evaluate(tree))
    except ZeroDivisionError:
        return "Error: cannot divide by zero."
    except OverflowError:
        return "Error: that result is too large to compute."
    except ValueError as exc:
        return f"Error: {exc}."


CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Evaluate one arithmetic expression and return the result. "
            "Use this for any ratio, growth rate, or multiple you need to compute "
            "rather than doing the arithmetic yourself. Pass numbers and the "
            "operators + - * / // % ** only -- no variable names, no function calls."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The arithmetic to evaluate, e.g. '(180 - 150) / 150 * 100'",
                }
            },
            "required": ["expression"],
        },
    },
}

In [20]:
# Stage 2: the same question, but now the model can do arithmetic.
STAGE_2_SCHEMAS = [CALCULATOR_SCHEMA]
STAGE_2_REGISTRY = {"calculator": calculator}

run_stage(client, MODEL, COMPANY, STAGE_2_SCHEMAS, STAGE_2_REGISTRY)

To provide a recommendation for Roche Holding AG (ROP.SW), I would need to consider several financial metrics such as the company's recent earnings performance, P/E ratio, revenue growth, and other relevant data. Since I cannot access real-time data, please provide any specific financial information or metrics for Roche Holding AG that you would like evaluated. This can include stock price, earnings, revenue, or any other key figures you have.

Tool calls, in order:
(none)

--- 1 model calls, 225 tokens


[AgentResult(answer="To provide a recommendation for Roche Holding AG (ROP.SW), I would need to consider several financial metrics such as the company's recent earnings performance, P/E ratio, revenue growth, and other relevant data. Since I cannot access real-time data, please provide any specific financial information or metrics for Roche Holding AG that you would like evaluated. This can include stock price, earnings, revenue, or any other key figures you have.", model_calls=1, tool_calls=[], stopped_early=False, trace=[], total_tokens=225)]

## Stage 3 · + web search

Current news. What changes, and what do you still not know about the numbers?

In [21]:
import re
from ddgs import DDGS

# The offline snapshot, embedded because Colab has no copy of the repository.
# It is used only when DuckDuckGo throttles us -- which it will, with everyone
# searching at once.
CACHE = {
    "generated_at": "",
    "ticker": "ROP.SW",
    "company": "Roche",
    "aliases": [
        "ROP.SW",
        "ROP",
        "Roche"
    ],
    "fundamentals": {},
    "search": [
        {
            "title": "PLACEHOLDER - run tools/make_snapshot.py to fetch real Roche coverage",
            "body": "This snapshot has not been generated yet. Until it is, the offline fallback returns this notice instead of real search results, so that nobody mistakes seeded text for genuine reporting on Roche.",
            "href": "https://example.invalid/placeholder"
        },
        {
            "title": "PLACEHOLDER - no cached news available",
            "body": "Regenerate before the lecture: uv run python tools/make_snapshot.py",
            "href": "https://example.invalid/placeholder"
        }
    ]
}


def load_cache():
    return CACHE


def _ddgs_text(query: str, max_results: int) -> list[dict]:
    """The live DuckDuckGo call. Separated out so tests can stand in for the network."""
    from ddgs import DDGS

    return list(DDGS().text(query, max_results=max_results))


def _format_results(results: list[dict]) -> str:
    lines = []
    for i, r in enumerate(results, start=1):
        lines.append(f"{i}. {r.get('title', '')}\n   {r.get('body', '')}\n   {r.get('href', '')}")
    return "\n".join(lines)


def web_search(query: str, max_results: int = 5) -> str:
    """Search the web and return the top results as text."""
    max_results = max(1, min(int(max_results or 5), 10))
    try:
        results = _ddgs_text(query, max_results)
    except Exception:
        results = []
    if results:
        return _format_results(results)

    # DuckDuckGo throttles when a whole classroom searches at once. That is an
    # expected outcome here, not an error: fall back to the cached snapshot and
    # say nothing, so a throttled student gets the same lesson as everyone else.
    #
    # But the snapshot is about ONE company. Serving its headlines for a query
    # about some other company would be inventing news, so the fallback only
    # applies when the query is actually about the company we cached.
    cache = load_cache()
    aliases = cache.get("aliases") or [cache.get("ticker", "")]
    # Whole words only: "ROG" must not match inside "programme".
    if not any(
        alias and re.search(rf"(?<!\w){re.escape(alias)}(?!\w)", query, re.IGNORECASE)
        for alias in aliases
    ):
        return (
            f"Error: the live search failed and there is no offline snapshot for "
            f"{query!r}. The cached snapshot only covers {cache.get('company') or cache.get('ticker')}."
        )
    cached = cache.get("search") or []
    if not cached:
        return (
            "Error: the live search failed and the offline snapshot holds no results. "
            "Ask whoever prepared this notebook to regenerate it."
        )
    return _format_results(cached[:max_results])


WEB_SEARCH_SCHEMA = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": (
            "Search the web and return the top results as text. Use this for current "
            "news, recent events, and analyst commentary -- anything that happened "
            "after your training data ends. Results are prose, not figures: use "
            "get_stock_data when you need numbers you can compute with."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to search for, e.g. 'Roche Q3 sales analyst reaction'",
                },
                "max_results": {
                    "type": "integer",
                    "description": "How many results to return. Defaults to 5.",
                },
            },
            "required": ["query"],
        },
    },
}

In [22]:
# Stage 3: add web search. Note that run_agent has not changed -- the list grew.
STAGE_3_SCHEMAS = [CALCULATOR_SCHEMA, WEB_SEARCH_SCHEMA]
STAGE_3_REGISTRY = {"calculator": calculator, "web_search": web_search}

run_stage(client, MODEL, COMPANY, STAGE_3_SCHEMAS, STAGE_3_REGISTRY)

Based on the recent information:

1. **Financial Performance**: Roche's third-quarter results for 2023 show a decline in sales by 14%, primarily due to a drop in COVID-19 product sales and competition from biosimilars. However, they saw a sales increase in new products, and core pharmaceutical sales rose by 7% in the same quarter. The overall annual sales growth for 2023 was 1%. 

2. **Market Reaction**: The stock experienced a significant drop in October 2023, declining by 5.1% upon a cautious forecast that disappointed investors who hoped for an upward revision. 

3. **Current Challenges**: Roche is facing challenges with a patent cliff looming, which could affect future revenue from key products like Xolair, and there are concerns about sustaining growth amid these pressures.

Considering these points, my recommendation for Roche Holding AG (ROP.SW) is a **Hold**. The stock has faced recent challenges and has shown some sales declines, but the company's ongoing development of new pr

[AgentResult(answer="Based on the recent information:\n\n1. **Financial Performance**: Roche's third-quarter results for 2023 show a decline in sales by 14%, primarily due to a drop in COVID-19 product sales and competition from biosimilars. However, they saw a sales increase in new products, and core pharmaceutical sales rose by 7% in the same quarter. The overall annual sales growth for 2023 was 1%. \n\n2. **Market Reaction**: The stock experienced a significant drop in October 2023, declining by 5.1% upon a cautious forecast that disappointed investors who hoped for an upward revision. \n\n3. **Current Challenges**: Roche is facing challenges with a patent cliff looming, which could affect future revenue from key products like Xolair, and there are concerns about sustaining growth amid these pressures.\n\nConsidering these points, my recommendation for Roche Holding AG (ROP.SW) is a **Hold**. The stock has faced recent challenges and has shown some sales declines, but the company's 

## Stage 4 · + yfinance

Structured market data with a date. Is it good enough now?

In [23]:
import yfinance
from datetime import date

# Securities are not interchangeable: Roche's Genussschein (ROP.SW) and its Inhaberaktie
# (RO.SW) trade at different prices. An agent that fetches whichever one it felt
# like has given you an answer about a different security than the one you asked
# about. The schema below tells the model so explicitly.

STOCK_FIELDS = [
    "currency",
    "currentPrice",
    "trailingPE",
    "forwardPE",
    "marketCap",
]


def _yf_info(ticker: str) -> dict:
    """The live Yahoo call. Separated out so tests can stand in for the network."""
    import yfinance

    return yfinance.Ticker(ticker).info


def _yf_history(ticker: str) -> list[float]:
    """One year of daily closing prices, oldest first. Separate for the same reason."""
    import yfinance

    return [float(c) for c in yfinance.Ticker(ticker).history(period="1y")["Close"].tolist()]


def _history_stats(closes: list[float]) -> dict:
    """1-year return and annualised volatility, computed here rather than by the model."""
    if len(closes) < 3 or closes[0] <= 0:
        return {}
    daily = [b / a - 1 for a, b in zip(closes, closes[1:])]
    mean = sum(daily) / len(daily)
    variance = sum((r - mean) ** 2 for r in daily) / (len(daily) - 1)
    return {
        "oneYearReturnPct": round((closes[-1] / closes[0] - 1) * 100, 2),
        "annualisedVolatilityPct": round(variance**0.5 * 252**0.5 * 100, 2),
    }


def _live_quote(ticker: str) -> dict:
    """Fetch the figures for one ticker. Raises if Yahoo has no price for it."""
    info = _yf_info(ticker)
    price = info.get("currentPrice")
    if price is None:
        raise ValueError(f"no price found for {ticker!r}")
    quote = {f: info[f] for f in STOCK_FIELDS if info.get(f) is not None}
    # Yahoo reports dividendYield as a fraction in some versions and a percentage in
    # others. The annual dividend per share divided by the price has no such ambiguity.
    if info.get("dividendRate") is not None and price:
        quote["dividendYieldPct"] = round(info["dividendRate"] / price * 100, 2)
    try:
        quote.update(_history_stats(_yf_history(ticker)))
    except Exception:
        pass  # a quote without history is still worth returning
    return quote


def get_stock_data(ticker: str) -> str:
    """Look up price, P/E, dividend yield, 1-year return and volatility, e.g. "ROP.SW"."""
    try:
        quote = _live_quote(ticker)
        as_of = date.today().isoformat()
    except Exception:
        # yfinance reads an undocumented endpoint that breaks without warning. The
        # cached snapshot keeps the lecture running; the student sees no difference.
        #
        # The snapshot covers ONE ticker, though. Filling an NESN.SW heading with the
        # cached Roche figures would attribute real numbers to the wrong security --
        # and ROG.SW vs RO.SW is the very share-class trap this lecture teaches. So the
        # fallback applies only when the ticker asked for is the one we cached.
        cache = load_cache()
        if ticker.strip().upper() != str(cache.get("ticker", "")).strip().upper():
            return (
                f"Error: the live lookup for {ticker!r} failed and there is no offline "
                f"snapshot for it (the snapshot covers {cache.get('ticker')} only). "
                f"Check the ticker includes its exchange, e.g. 'ROP.SW'. "
                f"Do not substitute figures from another ticker."
            )
        quote = cache.get("fundamentals") or {}
        as_of = cache.get("generated_at") or "unknown"
    lines = [f"{ticker} stock data (as of {as_of}):"]
    for field, value in quote.items():
        lines.append(f"{field}: {value}")
    if len(lines) == 1:
        # Yahoo failed and the snapshot was never generated. Say so rather than
        # hand back a bare heading: a silent empty result makes stage 4 look like
        # stage 1, and nobody in the room would know why.
        lines.append(
            "no data available -- the live lookup failed and the offline snapshot is "
            "empty. Ask whoever prepared this notebook to regenerate it."
        )
    return "\n".join(lines)


STOCK_DATA_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_stock_data",
        "description": (
            "Current price, P/E, dividend yield and 1-year return for a stock ticker. "
            "Use for any market figure. Also returns annualised volatility, the "
            "currency and the date of the data. Pass the ticker exactly as given "
            "to you and do not substitute a related one: ROP.SW (Roche "
            "Genussschein) and RO.SW (Roche Inhaberaktie) are different securities "
            "with different prices."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock symbol, optional with exchange",
                    "examples": ["ROP.SW", "AAPL"],
                }
            },
            "required": ["ticker"],
        },
    },
}

In [24]:
# Stage 4: real financial data. Same loop, one more function.
STAGE_4_SCHEMAS = [CALCULATOR_SCHEMA, WEB_SEARCH_SCHEMA, STOCK_DATA_SCHEMA]
STAGE_4_REGISTRY = {
    "calculator": calculator,
    "web_search": web_search,
    "get_stock_data": get_stock_data,
}

run_stage(client, MODEL, COMPANY, STAGE_4_SCHEMAS, STAGE_4_REGISTRY)

# Pick one figure out of that answer and trace it back: which tool produced it,
# and does it match what you find yourself?

c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inpl

**Recommendation: Hold**

**Justification:**

1. **Valuation**: The current price of a Roche share stands at CHF 400, with a trailing P/E of 26.09, which suggests the market has a positive outlook on its past earnings. However, the forward P/E of 18.13 indicates expectations of earnings growth, aligning well with future projections.

2. **Performance**: The stock has delivered a robust one-year return of 47.81%, which is a significant appreciation and may suggest limited short-term upside from this level without new business catalysts.

3. **Dividend Yield**: With a dividend yield of 2.45%, Roche provides a stable income stream which can be attractive to income-focused investors.

4. **Volatility**: The annualised volatility of 23.3% suggests moderate risk, which is typical for a blue-chip pharmaceutical company like Roche.

5. **Recent Developments**: Roche recently received a CE Mark for its cobas® smart e immunoanalyser, which enhances its medical testing portfolio. While this devel

[AgentResult(answer='**Recommendation: Hold**\n\n**Justification:**\n\n1. **Valuation**: The current price of a Roche share stands at CHF 400, with a trailing P/E of 26.09, which suggests the market has a positive outlook on its past earnings. However, the forward P/E of 18.13 indicates expectations of earnings growth, aligning well with future projections.\n\n2. **Performance**: The stock has delivered a robust one-year return of 47.81%, which is a significant appreciation and may suggest limited short-term upside from this level without new business catalysts.\n\n3. **Dividend Yield**: With a dividend yield of 2.45%, Roche provides a stable income stream which can be attractive to income-focused investors.\n\n4. **Volatility**: The annualised volatility of 23.3% suggests moderate risk, which is typical for a blue-chip pharmaceutical company like Roche.\n\n5. **Recent Developments**: Roche recently received a CE Mark for its cobas® smart e immunoanalyser, which enhances its medical te

## The calculator, revisited

Go back and look at stage 2 again.

Adding a calculator barely helped, and that surprises most people. The arithmetic was
never the problem -- the model can multiply. The problem was that it was multiplying
numbers it had invented. A calculator applied to a made-up price gives you a precisely
wrong answer, which is worse than an obviously wrong one, because it looks like work.

The calculator only became worth having at stage 4, once `get_stock_data` started
feeding it figures that were actually true. **A tool is only as good as what feeds it.**

That is the thing worth taking out of today. When someone shows you an agent, the
interesting question is not how many tools it has. It is where its facts come from, and
whether you can check them.

A few things we skipped, which matter in practice:

- **Cost.** Each stage printed its model-call count. Stage 4 costs several calls per
  question, not one -- a loop that calls tools is not a single API call.
- **Trust.** `web_search` returns whatever is on the internet, including things that are
  wrong. The agent has no way to tell a filing from a forum post.
- **Not advice.** None of this is investment advice, and a model with a data feed is
  still not a CFA.

## Bonus · if you finish early

Each cell below changes one thing. Run it, then compare the trace with stage 4.

In [25]:
# Bonus 1: the bank-grade system prompt (slide 21). Same question, same tools.
SYSTEM_PROMPT = BANK_SYSTEM_PROMPT
run_stage(client, MODEL, COMPANY, STAGE_4_SCHEMAS, STAGE_4_REGISTRY)

c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inpl

Here's a detailed briefing on Roche Holding AG (ROP.SW):

### Summary
Roche Holding AG's stock is currently trading at CHF 400.00, showing a strong one-year return of 47.81%. The stock's trailing P/E ratio stands at 26.09, which is quite high, indicating significant investor confidence or high expected growth. Roche's dividend yield of 2.45% offers a modest income stream for investors.

### Key Figures
| Metric              | Value          |
|---------------------|----------------|
| Current Price       | CHF 400.00     |
| Trailing P/E Ratio  | 26.09          |
| Forward P/E Ratio   | 18.14          |
| Market Cap          | CHF 318.7 billion |
| Dividend Yield      | 2.45%          |
| 1-Year Return       | 47.81%         |
| Annualised Volatility | 23.3%       |

### Risks
- **Market Volatility**: With an annualised volatility of 23.3%, Roche's stock experiences a relatively high degree of price fluctuations, which could lead to potential risks for investors.
- **High Valuation**: 

[AgentResult(answer="Here's a detailed briefing on Roche Holding AG (ROP.SW):\n\n### Summary\nRoche Holding AG's stock is currently trading at CHF 400.00, showing a strong one-year return of 47.81%. The stock's trailing P/E ratio stands at 26.09, which is quite high, indicating significant investor confidence or high expected growth. Roche's dividend yield of 2.45% offers a modest income stream for investors.\n\n### Key Figures\n| Metric              | Value          |\n|---------------------|----------------|\n| Current Price       | CHF 400.00     |\n| Trailing P/E Ratio  | 26.09          |\n| Forward P/E Ratio   | 18.14          |\n| Market Cap          | CHF 318.7 billion |\n| Dividend Yield      | 2.45%          |\n| 1-Year Return       | 47.81%         |\n| Annualised Volatility | 23.3%       |\n\n### Risks\n- **Market Volatility**: With an annualised volatility of 23.3%, Roche's stock experiences a relatively high degree of price fluctuations, which could lead to potential risks

In [ ]:
# Bonus 2: another Swiss share, with the plain prompt again. Only COMPANY changes.
# The offline fallback only knows Roche, so this needs live Yahoo and search.
SYSTEM_PROMPT = NAIVE_SYSTEM_PROMPT
run_stage(client, MODEL, "Nestlé S.A. (NESN.SW)", STAGE_4_SCHEMAS, STAGE_4_REGISTRY)

c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\bell\Documents\Code_bell\basic-agent\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inpl

### Recommendation: Hold

#### Justification:

1. **Valuation:**
   - **Trailing PE Ratio:** At 29.81, the trailing PE ratio suggests Nestlé is valued at a relatively high multiple compared to the market average. However, the forward PE of 18.08 indicates that earnings are expected to grow, which might justify the current high price if future performance meets expectations.

2. **Dividend Yield:**
   - **Yield of 3.6%:** This is an attractive yield, offering a solid return to income-focused investors, contributing to stability and income despite market fluctuations.

3. **Recent Performance:**
   - **1-Year Return:** With a 12.92% one-year return, Nestlé has shown pleasing performance, potentially reflecting investor confidence and sound management strategies.

4. **Recent News:**
   - According to the latest news, Nestlé's Q3 2026 results indicate a 2.8% real internal growth. This metric of actual product consumption growth reflects a positive company direction that aligns with the bu

[AgentResult(answer='### Recommendation: Hold\n\n#### Justification:\n\n1. **Valuation:**\n   - **Trailing PE Ratio:** At 29.81, the trailing PE ratio suggests Nestlé is valued at a relatively high multiple compared to the market average. However, the forward PE of 18.08 indicates that earnings are expected to grow, which might justify the current high price if future performance meets expectations.\n\n2. **Dividend Yield:**\n   - **Yield of 3.6%:** This is an attractive yield, offering a solid return to income-focused investors, contributing to stability and income despite market fluctuations.\n\n3. **Recent Performance:**\n   - **1-Year Return:** With a 12.92% one-year return, Nestlé has shown pleasing performance, potentially reflecting investor confidence and sound management strategies.\n\n4. **Recent News:**\n   - According to the latest news, Nestlé\'s Q3 2026 results indicate a 2.8% real internal growth. This metric of actual product consumption growth reflects a positive compa

In [27]:
# Bonus 3: ask for something nobody should promise. Try it with both system prompts.
SYSTEM_PROMPT = NAIVE_SYSTEM_PROMPT
run_stage(
    client, MODEL, COMPANY, STAGE_4_SCHEMAS, STAGE_4_REGISTRY,
    question="Guarantee me a 10 % return on Roche.",
)

Investments in stocks come with inherent risks, and it's not possible to guarantee a specific return, such as 10%, on any stock, including Roche. Stock prices are influenced by many factors, including market conditions, company performance, and broader economic trends, which can lead to fluctuations in returns.

If you're considering an investment in Roche or any other stock, it's essential to do thorough research or consult with a financial advisor to understand the risks and potential returns associated with your investment. Would you like me to provide some current data or analysis on Roche to help you make an informed decision?

Tool calls, in order:
(none)

--- 1 model calls, 469 tokens


[AgentResult(answer="Investments in stocks come with inherent risks, and it's not possible to guarantee a specific return, such as 10%, on any stock, including Roche. Stock prices are influenced by many factors, including market conditions, company performance, and broader economic trends, which can lead to fluctuations in returns.\n\nIf you're considering an investment in Roche or any other stock, it's essential to do thorough research or consult with a financial advisor to understand the risks and potential returns associated with your investment. Would you like me to provide some current data or analysis on Roche to help you make an informed decision?", model_calls=1, tool_calls=[], stopped_early=False, trace=[], total_tokens=469)]

In [ ]:
# ---------------------------------------------------------------------------
# Your turn.
#
# 1. Rewrite the prompt. Replace SYSTEM_PROMPT with the analyst you actually
#    want -- a credit analyst, a short seller, an ESG screener -- re-run
#    stage 4, and see what changes.
#
# 2. Add one tool. Fill in the three blanks below. You do not need to touch
#    run_agent: that is the point.
# ---------------------------------------------------------------------------


def my_tool(argument: str) -> str:
    """Say what your tool does -- the model reads this docstring's job in the
    schema below, not here."""
    # Return a STRING. Whatever you return is handed back to the model as text.
    return "..."


MY_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "my_tool",  # must match the function name in the registry below
        "description": (
            "Tell the model what this does and, just as importantly, when to "
            "reach for it instead of another tool."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "argument": {
                    "type": "string",
                    "description": "What this argument is",
                }
            },
            "required": ["argument"],
        },
    },
}


MY_SCHEMAS = STAGE_4_SCHEMAS + [MY_TOOL_SCHEMA]
MY_REGISTRY = {**STAGE_4_REGISTRY, "my_tool": my_tool}

run_stage(client, MODEL, COMPANY, MY_SCHEMAS, MY_REGISTRY)